In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Tiền xử lý dữ liệu

In [2]:
data = pd.read_csv('data/flight_delay.csv')
data.head()

,Unnamed: 0,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,...,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,DELAYED,DEP_HOUR,ARR_HOUR
0,4305278,2015,9,25,5,AA,2408,N3ABAA,DFW,ATL,...,0,NaN,NaN,NaN,NaN,NaN,NaN,0,18,21
1,650617,2015,2,12,4,US,1768,N822AW,CLT,MEM,...,0,NaN,0.0,0.0,3.0,26.0,0.0,1,20,21
2,2871905,2015,6,29,1,WN,1438,N784SW,BNA,CLE,...,0,NaN,0.0,0.0,0.0,0.0,15.0,1,21,23
3,3658069,2015,8,15,6,WN,3006,N787SA,CMH,MDW,...,0,NaN,0.0,0.0,0.0,63.0,0.0,1,17,18
4,2727299,2015,6,21,7,DL,1920,N967AT,EWR,ATL,...,0,NaN,NaN,NaN,NaN,NaN,NaN,0,11,14


In [3]:
data.columns

Index(['Unnamed: 0', 'YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE',
       'FLIGHT_NUMBER', 'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT',
       'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT',
       'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE',
       'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME',
       'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED', 'CANCELLATION_REASON',
       'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
       'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'DELAYED', 'DEP_HOUR',
       'ARR_HOUR'],
      dtype='str')

In [4]:
# loại bỏ các thuộc tính dư thừa
data = data.dropna(subset=['ARRIVAL_DELAY']) # loại bỏ các hàng rỗng
data = data.drop(columns=['Unnamed: 0', 'YEAR',
                          'FLIGHT_NUMBER', 'TAIL_NUMBER', 'SCHEDULED_TIME',
                          'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', # đã được thể hiện qua cột: 'DEP_HOUR', 'ARR_HOUR'
                          'TAXI_OUT', 'TAXI_IN', 'WHEELS_OFF', 'WHEELS_ON', 'ELAPSED_TIME', 'AIR_TIME',
                          'CANCELLATION_REASON', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY',
                          'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY'
                          ])
data.head()

,MONTH,DAY,DAY_OF_WEEK,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DEPARTURE_DELAY,DISTANCE,ARRIVAL_DELAY,DIVERTED,CANCELLED,DELAYED,DEP_HOUR,ARR_HOUR
0,9,25,5,AA,DFW,ATL,-4.0,731,-4.0,0,0,0,18,21
1,2,12,4,US,CLT,MEM,32.0,511,29.0,0,0,1,20,21
2,6,29,1,WN,BNA,CLE,20.0,448,15.0,0,0,1,21,23
3,8,15,6,WN,CMH,MDW,73.0,283,63.0,0,0,1,17,18
4,6,21,7,DL,EWR,ATL,16.0,746,-2.0,0,0,0,11,14


In [5]:
# kiểm tra tính chất các thuộc tính còn lại:
print('DELAYED==ARR>10:', ((data.ARRIVAL_DELAY>10).astype(int)==data.DELAYED).mean())
print('DELAYED==ARR>0:', ((data.ARRIVAL_DELAY>0).astype(int)==data.DELAYED).mean())
print('DELAYED==DEP>10:', ((data.DEPARTURE_DELAY>10).astype(int)==data.DELAYED).mean())
print('DELAYED==DEP>0:', ((data.DEPARTURE_DELAY>0).astype(int)==data.DELAYED).mean())
print(data[data.CANCELLED==1].DELAYED.value_counts())
print(data[data.DIVERTED==1].DELAYED.value_counts())
print(data.ORIGIN_AIRPORT.nunique(), data.AIRLINE.nunique())
print(data.groupby('DELAYED').DEPARTURE_DELAY.mean())
print(data.groupby('DEP_HOUR').DELAYED.mean().round(2).to_dict())

DELAYED==ARR>10: 0.9898487082766431
DELAYED==ARR>0: 0.8660330935369023
DELAYED==DEP>10: 0.8965399248837
DELAYED==DEP>0: 0.7942652534156734
Series([], Name: count, dtype: int64)
Series([], Name: count, dtype: int64)
620 14
DELAYED
0    -1.725613
1    46.267020
Name: DEPARTURE_DELAY, dtype: float64
{0: 0.23, 1: 0.28, 2: 0.16, 3: 0.27, 4: 0.26, 5: 0.1, 6: 0.12, 7: 0.15, 8: 0.17, 9: 0.18, 10: 0.19, 11: 0.21, 12: 0.22, 13: 0.24, 14: 0.26, 15: 0.26, 16: 0.28, 17: 0.3, 18: 0.31, 19: 0.31, 20: 0.31, 21: 0.29, 22: 0.27, 23: 0.24}


#### Things to notice:
- Các chuyến bay bị trễ có sự tương quan đến 2 cột "ARRIVAL_DELAY" và "DEPARTURE_DELAY" (98% phụ thuộc vào ARRIVAL_DELAY > 10)
- Những chuyến bay bị hủy (CANCELLED) hoặc bị hoãn (DIVERTED) không được gán nhãn là bị DELAYED -> Coi nó là dữ liệu không liên quan do ARRIVAL_DELAY là không xác định
- Cột ORIGIN_AIRPORT có 640 giá trị khác nhau, điều này có thể ảnh hưởng đến việc xây dựng mô hình do bùng nổ số chiều
- Thời gian khởi hành trễ trung bình khi được gán nhãn là trễ là 46 phút, ngược lại là sớm hơn ~2 phút
- Dữ liệu dạng thời gian là dạng chu kỳ (gán là 23:00 và 00:00), cần được xử lý

In [6]:
# loại bỏ các dòng tồn tại nhãn CANCELLED hoặc DIVERTED
data = data.drop(data[(data["CANCELLED"] == 1) | (data["DIVERTED"] == 1)].index).reset_index(drop=True)
data = data.drop(columns=["CANCELLED", "DIVERTED", "ARRIVAL_DELAY", "DEPARTURE_DELAY"])

In [7]:
# Giảm thiểu số lượng giá trị khác nhau trong 2 cột ORIGIN_AIRPORT, DESTINATION_AIRPORT
# bằng cách chỉ lấy 20 giá trị phổ biến nhất, còn lại sửa là Other

top_origin = data["ORIGIN_AIRPORT"].value_counts().nlargest(20).index
data["ORIGIN_AIRPORT"] = data["ORIGIN_AIRPORT"].where(
    data["ORIGIN_AIRPORT"].isin(top_origin), "Other"
)

top_dest = data["DESTINATION_AIRPORT"].value_counts().nlargest(20).index
data["DESTINATION_AIRPORT"] = data["DESTINATION_AIRPORT"].where(
    data["DESTINATION_AIRPORT"].isin(top_dest), "Other"
)

In [8]:
# Xử lý dữ liệu dạng thời gian bằng cách đưa về dạng chu kỳ sin-cos
def add_cyclical(df, col, period):
    df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
    df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)

add_cyclical(data, "DAY_OF_WEEK", 7)
add_cyclical(data, "DEP_HOUR", 24)
add_cyclical(data, "ARR_HOUR", 24)

data = data.drop(columns=["DAY_OF_WEEK", "DEP_HOUR", "ARR_HOUR", "DAY"]) # bỏ luôn cột DAY vì nó khó xử lý :vv

In [9]:
data.head()

,MONTH,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DISTANCE,DELAYED,DAY_OF_WEEK_sin,DAY_OF_WEEK_cos,DEP_HOUR_sin,DEP_HOUR_cos,ARR_HOUR_sin,ARR_HOUR_cos
0,9,AA,DFW,ATL,731,0,-9.749279e-01,-0.222521,-1.000000,-1.836970e-16,-0.707107,7.071068e-01
1,2,US,CLT,Other,511,1,-4.338837e-01,-0.900969,-0.866025,5.000000e-01,-0.707107,7.071068e-01
2,6,WN,Other,Other,448,1,7.818315e-01,0.623490,-0.707107,7.071068e-01,-0.258819,9.659258e-01
3,8,WN,Other,Other,283,1,-7.818315e-01,0.623490,-0.965926,-2.588190e-01,-1.000000,-1.836970e-16
4,6,DL,EWR,ATL,746,0,-2.449294e-16,1.000000,0.258819,-9.659258e-01,-0.500000,-8.660254e-01


In [10]:
# Xử lý dữ liệu rời rạc
cat_cols = ["AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
data = pd.get_dummies(data, columns=cat_cols, dtype=int) # one-hot encoding

In [11]:
# final check before export to cleaned dataset file
print(data.shape)
print(data.isna().sum().sum())      # should be 0
print(data["DELAYED"].mean())       # ~0.23
print(data.dtypes.value_counts())   # should be all ints/floats, no object columns

(245486, 65)
0
0.23066488516656755
int64      59
float64     6
Name: count, dtype: int64


In [12]:
data.to_csv("data/flight_delay_preprocessed.csv", index=False)